# NLP Text Representation

This notebook covers the topics listed in `list_of_topics` using simple and modular examples.



Topics included:

- One-hot encoding

- Label encoding

- Ordinal encoding

- Bag of words

- Bag of bi-gram

- Bag of n-gram

- TF-IDF

- Word2Vec and GloVe-style embeddings

- BERT and RoBERTa



Each section uses a very small dataset so the outputs are easy to understand.

In [1]:
# Import the libraries used in the notebook.

import numpy as np

import pandas as pd

import torch

from IPython.display import display

from gensim.models import Word2Vec

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder

from transformers import BertConfig, BertModel, RobertaConfig, RobertaModel



# Set seeds so the random examples stay reproducible.

np.random.seed(42)

torch.manual_seed(42)



# Create a tiny text corpus for the examples.

documents = [

    "I love learning NLP",

    "NLP makes language tasks easier",

    "I enjoy building simple NLP models",

]



token_categories = pd.DataFrame({"token": ["nlp", "ai", "ml", "nlp"]})

sentiment_levels = pd.DataFrame({"sentiment": ["negative", "neutral", "positive", "neutral"]})

tokenized_corpus = [sentence.lower().split() for sentence in documents]



print("Documents:")

for index, sentence in enumerate(documents, start=1):

    print(f"{index}. {sentence}")

Documents:
1. I love learning NLP
2. NLP makes language tasks easier
3. I enjoy building simple NLP models


## 1. One-Hot, Label, and Ordinal Encoding

These are basic encoding methods that convert categories into numbers.



- One-hot encoding creates one binary column per category.

- Label encoding maps each category to an integer.

- Ordinal encoding is used when the categories have a natural order.

In [2]:
# One-hot encoding for token categories.

one_hot_encoder = OneHotEncoder(sparse_output=False)

one_hot_result = one_hot_encoder.fit_transform(token_categories[["token"]])

one_hot_df = pd.DataFrame(

    one_hot_result,

    columns=one_hot_encoder.get_feature_names_out(["token"]),

)



# Label encoding for the same token categories.

label_encoder = LabelEncoder()

label_values = label_encoder.fit_transform(token_categories["token"])

label_df = pd.DataFrame(

    {"token": token_categories["token"], "label_encoded": label_values}

)



# Ordinal encoding for ordered sentiment levels.

ordinal_encoder = OrdinalEncoder(categories=[["negative", "neutral", "positive"]])

ordinal_values = ordinal_encoder.fit_transform(sentiment_levels[["sentiment"]])

ordinal_df = pd.DataFrame(

    {

        "sentiment": sentiment_levels["sentiment"],

        "ordinal_encoded": ordinal_values.ravel().astype(int),

    }

)



print("One-hot encoding:")

display(one_hot_df)



print("Label encoding:")

display(label_df)



print("Ordinal encoding:")

display(ordinal_df)

One-hot encoding:


,token_ai,token_ml,token_nlp
0,0.0,0.0,1.0
1,1.0,0.0,0.0
2,0.0,1.0,0.0
3,0.0,0.0,1.0


Label encoding:


,token,label_encoded
0,nlp,2
1,ai,0
2,ml,1
3,nlp,2


Ordinal encoding:


,sentiment,ordinal_encoded
0,negative,0
1,neutral,1
2,positive,2
3,neutral,1


## 2. Bag of Words

Bag of Words converts text into word-count vectors.



It ignores grammar and word order, and only keeps track of how many times each word appears in each document.

In [3]:
# Count each word in the documents.

bow_vectorizer = CountVectorizer()

bow_matrix = bow_vectorizer.fit_transform(documents)

bow_df = pd.DataFrame(

    bow_matrix.toarray(),

    columns=bow_vectorizer.get_feature_names_out(),

    index=[f"Doc {index}" for index in range(1, len(documents) + 1)],

)



display(bow_df)

,building,easier,enjoy,language,learning,love,makes,models,nlp,simple,tasks
Doc 1,0,0,0,0,1,1,0,0,1,0,0
Doc 2,0,1,0,1,0,0,1,0,1,0,1
Doc 3,1,0,1,0,0,0,0,1,1,1,0


## 3. Bag of Bi-gram and Bag of N-gram

N-grams capture short pieces of word order by looking at consecutive words together.



- A **unigram** uses single words only.

- A **bi-gram** uses pairs of consecutive words.

- A **trigram** uses groups of three consecutive words.

- An **n-gram** is the general form, where $n$ can be 1, 2, 3, or more.



Important note:



- When $n = 1$, the n-gram model is the same as the **Bag of Words unigram representation**.

- When $n > 1$, the model starts capturing some local word order information.

In [4]:
# Build a bag of bi-grams.

bigram_vectorizer = CountVectorizer(ngram_range=(2, 2))

bigram_matrix = bigram_vectorizer.fit_transform(documents)

bigram_df = pd.DataFrame(

    bigram_matrix.toarray(),

    columns=bigram_vectorizer.get_feature_names_out(),

    index=[f"Doc {index}" for index in range(1, len(documents) + 1)],

)



# Build a bag of n-grams using unigram, bigram, and trigram ranges.

ngram_vectorizer = CountVectorizer(ngram_range=(1, 3))

ngram_matrix = ngram_vectorizer.fit_transform(documents)

ngram_df = pd.DataFrame(

    ngram_matrix.toarray(),

    columns=ngram_vectorizer.get_feature_names_out(),

    index=[f"Doc {index}" for index in range(1, len(documents) + 1)],

)



print("Bag of bi-grams:")

display(bigram_df)



print("Bag of n-grams up to trigrams:")

display(ngram_df)

Bag of bi-grams:


,building simple,enjoy building,language tasks,learning nlp,love learning,makes language,nlp makes,nlp models,simple nlp,tasks easier
Doc 1,0,0,0,1,1,0,0,0,0,0
Doc 2,0,0,1,0,0,1,1,0,0,1
Doc 3,1,1,0,0,0,0,0,1,1,0


Bag of n-grams up to trigrams:


,building,building simple,building simple nlp,easier,enjoy,enjoy building,enjoy building simple,language,language tasks,language tasks easier,...,models,nlp,nlp makes,nlp makes language,nlp models,simple,simple nlp,simple nlp models,tasks,tasks easier
Doc 1,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
Doc 2,0,0,0,1,0,0,0,1,1,1,...,0,1,1,1,0,0,0,0,1,1
Doc 3,1,1,1,0,1,1,1,0,0,0,...,1,1,0,0,1,1,1,1,0,0


## 4. TF-IDF

TF-IDF stands for Term Frequency and Inverse Document Frequency.



- **TF** measures how often a word appears in a single document.

- **IDF** measures how rare or common that word is across the full corpus.



Simple intuition:



- If a word is **rare in the corpus** and appears **many times in one document**, its TF-IDF score becomes **high**.

- If a word is **very common across many documents**, its IDF becomes **low**, so its TF-IDF score stays **low or moderate** even if it appears many times.



TF-IDF still creates a sparse vector like Bag of Words, but it gives more importance to informative words and less importance to very common words.

In [5]:
# Compute TF-IDF scores for each document.

tfidf_vectorizer = TfidfVectorizer()

tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

tfidf_df = pd.DataFrame(

    tfidf_matrix.toarray().round(3),

    columns=tfidf_vectorizer.get_feature_names_out(),

    index=[f"Doc {index}" for index in range(1, len(documents) + 1)],

)



display(tfidf_df)

,building,easier,enjoy,language,learning,love,makes,models,nlp,simple,tasks
Doc 1,0.00,0.00,0.00,0.00,0.652,0.652,0.00,0.00,0.385,0.00,0.00
Doc 2,0.00,0.48,0.00,0.48,0.000,0.000,0.48,0.00,0.283,0.00,0.48
Doc 3,0.48,0.00,0.48,0.00,0.000,0.000,0.00,0.48,0.283,0.48,0.00


## 5. Word2Vec and GloVe

These are dense word embedding methods.



- Word2Vec learns vectors from local context in the training corpus.

- GloVe is also a static embedding method, but it is usually learned from global co-occurrence statistics.



To keep this notebook fully offline and simple:

- Word2Vec is trained on the tiny sample corpus.

- GloVe is shown with a small fixed embedding table that behaves like pre-trained static vectors.

In [6]:
# Train a tiny Word2Vec model on the corpus.

word2vec_model = Word2Vec(

    sentences=tokenized_corpus,

    vector_size=10,

    window=2,

    min_count=1,

    workers=1,

    sg=1,

    epochs=200,

    seed=42,

)



word2vec_vector = pd.DataFrame(

    [word2vec_model.wv["nlp"]],

    index=["nlp"],

    columns=[f"dim_{index}" for index in range(1, 11)],

).round(4)



# Use a small fixed GloVe-style embedding dictionary.

glove_embeddings = {

    "nlp": np.array([0.90, 0.10, 0.20, 0.70]),

    "language": np.array([0.88, 0.12, 0.18, 0.68]),

    "learning": np.array([0.40, 0.75, 0.60, 0.20]),

}



def cosine_similarity(vector_a, vector_b):

    numerator = np.dot(vector_a, vector_b)

    denominator = np.linalg.norm(vector_a) * np.linalg.norm(vector_b)

    return float(numerator / denominator)



glove_df = pd.DataFrame(glove_embeddings).T

glove_df.columns = [f"dim_{index}" for index in range(1, glove_df.shape[1] + 1)]

glove_similarity = cosine_similarity(glove_embeddings["nlp"], glove_embeddings["language"])



print("Word2Vec vector for 'nlp':")

display(word2vec_vector)



print("GloVe-style embedding table:")

display(glove_df)



print(f"Cosine similarity between 'nlp' and 'language': {glove_similarity:.4f}")

Word2Vec vector for 'nlp':


,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,dim_9,dim_10
nlp,-0.0835,0.0583,0.0323,-0.0114,-0.0126,0.0738,-0.085,0.0412,-0.0601,-0.0827


GloVe-style embedding table:


,dim_1,dim_2,dim_3,dim_4
nlp,0.90,0.10,0.20,0.70
language,0.88,0.12,0.18,0.68
learning,0.40,0.75,0.60,0.20


Cosine similarity between 'nlp' and 'language': 0.9997


## 6. BERT and RoBERTa

BERT and RoBERTa are contextual embedding models built with transformers.



Unlike Word2Vec and GloVe, the representation of a token depends on the surrounding words.



To make the notebook run without internet access, this example builds small local BERT and RoBERTa models from configuration only. The outputs are real transformer tensors, but the weights are randomly initialized rather than pre-trained.

In [7]:
# Build small local BERT and RoBERTa models.

bert_config = BertConfig(

    vocab_size=50,

    hidden_size=32,

    num_hidden_layers=2,

    num_attention_heads=4,

    intermediate_size=64,

)

bert_model = BertModel(bert_config)



roberta_config = RobertaConfig(

    vocab_size=50,

    hidden_size=32,

    num_hidden_layers=2,

    num_attention_heads=4,

    intermediate_size=64,

)

roberta_model = RobertaModel(roberta_config)



# Create a short token id sequence and an attention mask.

sample_input_ids = torch.tensor([[1, 7, 9, 11, 2]])

sample_attention_mask = torch.ones_like(sample_input_ids)



bert_output = bert_model(

    input_ids=sample_input_ids,

    attention_mask=sample_attention_mask,

)

roberta_output = roberta_model(

    input_ids=sample_input_ids,

    attention_mask=sample_attention_mask,

)



shape_df = pd.DataFrame(

    {

        "model": ["BERT", "RoBERTa"],

        "last_hidden_state_shape": [

            tuple(bert_output.last_hidden_state.shape),

            tuple(roberta_output.last_hidden_state.shape),

        ],

        "vector_size": [

            bert_output.pooler_output.shape[-1],

            roberta_output.last_hidden_state.shape[-1],

        ],

    }

)



display(shape_df)



print("BERT pooled output, first 5 values:")

print(bert_output.pooler_output[0, :5].detach().numpy().round(4))



print("RoBERTa first-token output, first 5 values:")

print(roberta_output.last_hidden_state[0, 0, :5].detach().numpy().round(4))

,model,last_hidden_state_shape,vector_size
0,BERT,"(1, 5, 32)",32
1,RoBERTa,"(1, 5, 32)",32


BERT pooled output, first 5 values:
[-0.1239 -0.1879 -0.0312 -0.0846  0.0228]
RoBERTa first-token output, first 5 values:
[-0.9738  0.2599 -0.3289 -0.3283  1.6409]
